# M2: FP32 vs INT8 TFLite (same test set, same tool)
Upload `best.pt` and `nutsbolts_dataset_320.zip` to /content first. Runtime -> T4 GPU is fine (export runs on CPU).

In [ ]:
!pip -q install ultralytics
!unzip -q -o /content/nutsbolts_dataset_320.zip -d /content/
!ls /content /content/ds3

In [ ]:
from ultralytics import YOLO
import os, json
data = '/content/ds3/data.yaml'
m = YOLO('/content/best.pt')
r32 = m.val(data=data, imgsz=320, split='test', batch=1, device='cpu', plots=False)
res = {'fp32_pt': dict(map50=float(r32.box.map50), map5095=float(r32.box.map), p=float(r32.box.mp), r=float(r32.box.mr), size_mb=os.path.getsize('/content/best.pt')/1e6)}
print(res)

In [ ]:
p = m.export(format='tflite', int8=True, imgsz=320, data=data, fraction=0.5)
print('EXPORTED', p)
!ls -la /content/best_saved_model /content/best_litert_model 2>/dev/null

In [ ]:
import glob
cands = glob.glob('/content/best_saved_model/*int8*.tflite') + glob.glob('/content/best_litert_model/*int8*.tflite') + glob.glob('/content/**/*int8*.tflite', recursive=True)
print(cands)
tfl = cands[0]
q = YOLO(tfl, task='detect')
r8 = q.val(data=data, imgsz=320, split='test', batch=1, device='cpu', plots=False)
res['int8_tflite'] = dict(map50=float(r8.box.map50), map5095=float(r8.box.map), p=float(r8.box.mp), r=float(r8.box.mr), size_mb=os.path.getsize(tfl)/1e6, file=tfl)
print(json.dumps(res, indent=1))
json.dump(res, open('/content/m2_int8_results.json','w'), indent=1)

In [ ]:
import shutil
shutil.copy(tfl, '/content/best_int8.tflite')
!cd /content && zip -q m2_int8.zip best_int8.tflite m2_int8_results.json
from google.colab import files
files.download('/content/m2_int8.zip')
